In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
model = models.resnet50(pretrained=False)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last block of layer4 (same as our training)
for param in model.layer4[-1].parameters():
    param.requires_grad = True

# Custom classifier
num_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(0.7),
    nn.Linear(256, 2)
)

# Load weights
model.load_state_dict(torch.load("gla_p1_75.pth", map_location=device))

model = model.to(device)
model.eval()

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
img_path = "test.jpg"  

image = Image.open(img_path).convert("RGB")
image = transform(image)
image = image.unsqueeze(0).to(device)

In [ ]:
import torch.nn.functional as F

with torch.no_grad():
    outputs = model(image)
    probs = F.softmax(outputs, dim=1)
    confidence, predicted = torch.max(probs, 1)

classes = ['glaucoma', 'normal']  

print("Prediction:", classes[predicted.item()])
print("Confidence:", confidence.item())